In [0]:
# ============================================================
# UNITY CATALOG VOLUME BACKUP FRAMEWORK
# Setup Notebook
# ============================================================

BACKUP_CATALOG = "backup_admin"
BACKUP_SCHEMA = "volume_protection"
BACKUP_VOLUME = "backups"

POLICY_TABLE = f"{BACKUP_CATALOG}.{BACKUP_SCHEMA}.volume_backup_policy"
MANIFEST_TABLE = f"{BACKUP_CATALOG}.{BACKUP_SCHEMA}.volume_backup_manifest"
RUN_TABLE = f"{BACKUP_CATALOG}.{BACKUP_SCHEMA}.volume_backup_run"

BACKUP_VOLUME_PATH = (
    f"/Volumes/{BACKUP_CATALOG}/"
    f"{BACKUP_SCHEMA}/{BACKUP_VOLUME}"
)

print("Backup framework configuration")
print("--------------------------------")
print(f"Catalog        : {BACKUP_CATALOG}")
print(f"Schema         : {BACKUP_SCHEMA}")
print(f"Policy table   : {POLICY_TABLE}")
print(f"Manifest table : {MANIFEST_TABLE}")
print(f"Run table      : {RUN_TABLE}")
print(f"Backup volume  : {BACKUP_VOLUME_PATH}")


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS backup_admin;

CREATE SCHEMA IF NOT EXISTS backup_admin.volume_protection
COMMENT 'Metadata and storage objects for Unity Catalog volume backup and recovery framework';

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS backup_admin.volume_protection.backups
COMMENT 'Seven-day file backup storage for protected Unity Catalog volumes';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS backup_admin.volume_protection.volume_backup_policy
(
    policy_id STRING NOT NULL,

    source_catalog STRING NOT NULL,
    source_schema STRING NOT NULL,
    source_volume STRING NOT NULL,

    backup_enabled BOOLEAN NOT NULL,

    backup_priority STRING,

    retention_days INT NOT NULL,

    backup_frequency STRING,

    created_at TIMESTAMP,
    created_by STRING,

    updated_at TIMESTAMP,
    updated_by STRING,

    description STRING
)
USING DELTA
COMMENT 'Configuration defining which Unity Catalog volumes are protected and their backup retention policies';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS backup_admin.volume_protection.volume_backup_manifest
(
    snapshot_id STRING NOT NULL,

    policy_id STRING,

    snapshot_timestamp TIMESTAMP NOT NULL,

    source_catalog STRING NOT NULL,
    source_schema STRING NOT NULL,
    source_volume STRING NOT NULL,

    source_relative_path STRING NOT NULL,
    source_full_path STRING NOT NULL,

    backup_full_path STRING,

    file_name STRING,

    file_size_bytes BIGINT,

    source_modification_time TIMESTAMP,

    file_checksum STRING,

    change_type STRING,

    backup_status STRING,

    error_message STRING,

    created_at TIMESTAMP
)
USING DELTA
COMMENT 'File-level snapshot manifest for Unity Catalog volume backup and point-in-time recovery';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS backup_admin.volume_protection.volume_backup_run
(
    run_id STRING NOT NULL,

    start_timestamp TIMESTAMP NOT NULL,

    end_timestamp TIMESTAMP,

    run_status STRING,

    protected_volume_count INT,

    scanned_file_count BIGINT,

    new_file_count BIGINT,

    modified_file_count BIGINT,

    unchanged_file_count BIGINT,

    deleted_file_count BIGINT,

    backed_up_file_count BIGINT,

    failed_file_count BIGINT,

    bytes_backed_up BIGINT,

    error_message STRING,

    created_at TIMESTAMP
)
USING DELTA
COMMENT 'Operational audit history for Unity Catalog volume backup workflow executions';

In [0]:
%sql
MERGE INTO backup_admin.volume_protection.volume_backup_policy AS target

USING
(
    SELECT

        'POLICY_APP_ARTIFACTS' AS policy_id,

        'royalinsurance' AS source_catalog,

        'edw' AS source_schema,

        'app_artifacts' AS source_volume,

        true AS backup_enabled,

        'CRITICAL' AS backup_priority,

        7 AS retention_days,

        'DAILY' AS backup_frequency,

        current_timestamp() AS created_at,

        current_user() AS created_by,

        current_timestamp() AS updated_at,

        current_user() AS updated_by,

        'Backup policy for Royal Insurance application artifacts'
            AS description

) AS source

ON target.policy_id = source.policy_id

WHEN MATCHED THEN
UPDATE SET

    target.backup_enabled =
        source.backup_enabled,

    target.backup_priority =
        source.backup_priority,

    target.retention_days =
        source.retention_days,

    target.backup_frequency =
        source.backup_frequency,

    target.updated_at =
        source.updated_at,

    target.updated_by =
        source.updated_by,

    target.description =
        source.description

WHEN NOT MATCHED THEN
INSERT *;

In [0]:
%sql
SELECT
    policy_id,
    source_catalog,
    source_schema,
    source_volume,
    backup_enabled,
    backup_priority,
    retention_days,
    backup_frequency,
    created_at,
    updated_at

FROM backup_admin.volume_protection.volume_backup_policy

ORDER BY
    backup_priority,
    source_catalog,
    source_schema,
    source_volume;

In [0]:
print("Backup volume contents:")
print(BACKUP_VOLUME_PATH)

display(
    dbutils.fs.ls(
        BACKUP_VOLUME_PATH
    )
)

In [0]:
%sql
SHOW TABLES
IN backup_admin.volume_protection;

In [0]:
%sql
SHOW TABLES
IN backup_admin.volume_protection;

In [0]:
policy_count = spark.sql(f"""
    SELECT COUNT(*) AS cnt
    FROM {POLICY_TABLE}
    WHERE backup_enabled = true
""").first()["cnt"]

print("=" * 60)
print("UNITY CATALOG VOLUME PROTECTION FRAMEWORK")
print("=" * 60)

print(f"""
Setup completed successfully.

Control tables
--------------
Policy   : {POLICY_TABLE}
Manifest : {MANIFEST_TABLE}
Runs     : {RUN_TABLE}

Backup storage
--------------
{BACKUP_VOLUME_PATH}

Protection
----------
Enabled volumes : {policy_count}
Default retention: 7 days

Next notebook:
02_volume_backup_job
""")